In [1]:
import json
from pathlib import Path
import pandas as pd

DATA = Path("../open-data/data")

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


matches vs 360 files per competition-season

In [2]:
comps = pd.DataFrame(load(DATA / "competitions.json"))
comps_360 = comps[comps["match_available_360"].notna()]

ids_360 = {int(p.stem) for p in (DATA / "three-sixty").glob("*.json")}

rows = []
for c in comps_360.itertuples():
    matches = load(DATA / "matches" / str(c.competition_id) / f"{c.season_id}.json")
    match_ids = {m["match_id"] for m in matches}
    rows.append({
        "competition": c.competition_name,
        "season": c.season_name,
        "matches": len(match_ids),
        "with_360_file": len(match_ids & ids_360),
    })

summary = pd.DataFrame(rows)
summary

,competition,season,matches,with_360_file
0,1. Bundesliga,2023/2024,34,34
1,African Cup of Nations,2023,52,1
2,FIFA World Cup,2022,64,64
3,La Liga,2020/2021,35,35
4,Ligue 1,2022/2023,32,32
5,Ligue 1,2021/2022,26,26
6,Major League Soccer,2023,6,6
7,UEFA Euro,2024,51,51
8,UEFA Euro,2020,51,51
9,UEFA Women's Euro,2025,31,31


One match, keys and a player entry

In [3]:
match_id = sorted(ids_360)[0]

events = load(DATA / "events" / f"{match_id}.json")
frames = load(DATA / "three-sixty" / f"{match_id}.json")


print(len(events), "events,", len(frames), "frames")
print("frame keys:", list(frames[0].keys()))
print("one player entry:", frames[0]["freeze_frame"][0])
print("event keys:", list(events[0].keys()))

4160 events, 3914 frames
frame keys: ['event_uuid', 'visible_area', 'freeze_frame']
one player entry: {'teammate': False, 'actor': False, 'keeper': False, 'location': [52.273950315181956, 38.42847753920154]}
event keys: ['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type', 'possession', 'possession_team', 'play_pattern', 'team', 'duration', 'tactics']


Join a shot to its frame

In [4]:
shots = [e for e in events if e["type"]["name"] == "Shot"]
frame_by_event = {f["event_uuid"]: f for f in frames}

shot = shots[0]
print("This shot has frame", shot["id"] in frame_by_event)

with_frame = sum(s["id"] in frame_by_event for s in shots)
print(f"{with_frame} of {len(shots)} shots have a 360 frame")

This shot has frame True
15 of 15 shots have a 360 frame


In [5]:
rows, failed = [], []
for match_id in sorted(ids_360):
    try:
        events = load(DATA / "events" / f"{match_id}.json")
        frames = load(DATA / "three-sixty" / f"{match_id}.json")
    except json.JSONDecodeError as err:
        failed.append((match_id, str(err)[:60]))   # record it and keep going
        continue
    frame_ids = {f["event_uuid"] for f in frames}
    event_ids = {e["id"] for e in events}
    shots = [e for e in events if e["type"]["name"] == "Shot"]
    rows.append({
        "match_id": match_id,
        "shots": len(shots),
        "shots_with_frame": sum(s["id"] in frame_ids for s in shots),
        "frames": len(frame_ids),
        "frames_matching_an_event": len(frame_ids & event_ids),
    })

per_match = pd.DataFrame(rows)
print("failed to load:", failed)
print(per_match[["shots", "shots_with_frame"]].sum())
per_match.sort_values("shots_with_frame").head(12)

failed to load: [(3845506, "Expecting ',' delimiter: line 92794 column 3 (char 2637824)")]
shots               11033
shots_with_frame    10483
dtype: int64


,match_id,shots,shots_with_frame,frames,frames_matching_an_event
321,3895309,21,0,3280,0
316,3895266,22,0,3615,0
304,3895158,30,0,3318,0
142,3837706,20,0,3548,0
342,3923880,33,0,1,1
242,3877170,33,0,3097,0
243,3877194,22,0,3174,0
241,3877115,21,0,3808,0
209,3857289,9,9,2751,2751
109,3802983,10,10,3503,3503


In [6]:
# matches where at least one shot can be joined to a 360 frame
usable_ids = per_match.loc[per_match["shots_with_frame"] > 0, "match_id"]

rows = []
for match_id in usable_ids:
    events = load(DATA / "events" / f"{match_id}.json")
    frame_ids = {f["event_uuid"] for f in load(DATA / "three-sixty" / f"{match_id}.json")}
    for e in events:
        if e["type"]["name"] != "Shot":
            continue
        s = e["shot"]
        rows.append({
            "shot_id": e["id"],
            "match_id": match_id,
            "period": e["period"],
            "minute": e["minute"],
            "team": e["team"]["name"],
            "player": e["player"]["name"],
            "x": e["location"][0],
            "y": e["location"][1],
            "shot_type": s["type"]["name"],
            "outcome": s["outcome"]["name"],
            "goal": int(s["outcome"]["name"] == "Goal"),
            "has_frame": e["id"] in frame_ids,
        })
shots_df = pd.DataFrame(rows)

model = shots_df[
    shots_df["has_frame"]
    & (shots_df["period"] != 5)               # period 5 = penalty shootout
    & (shots_df["shot_type"] != "Penalty")
]
print(len(shots_df), "->", len(model), "shots | goal rate:", round(model["goal"].mean(), 3))

10831 -> 10233 shots | goal rate: 0.102


In [7]:
pens = shots_df[shots_df["shot_type"] == "Penalty"]
print(pens["has_frame"].mean(), pens["goal"].mean())          # penalties
print(shots_df[shots_df["period"] == 5]["has_frame"].mean())  # shootout shots
print(model["goal"].mean())                                   # everything else

0.7692307692307693 0.6738461538461539
0.6338797814207651
0.1015342519300303


1. Penalties are shot from a fixed spot with a goal rate around 75%

Features for ANN

In [8]:
usable_ids = per_match.loc[per_match["shots_with_frame"] > 0, "match_id"]

rows = []
for match_id in usable_ids:
    events = load(DATA / "events" / f"{match_id}.json")
    frame_ids = {f["event_uuid"] for f in load(DATA / "three-sixty" / f"{match_id}.json")}
    for e in events:
        if e["type"]["name"] != "Shot":
            continue
        s = e["shot"]
        rows.append({
            "shot_id": e["id"],
            "match_id": match_id,
            "period": e["period"],
            "minute": e["minute"],
            "team": e["team"]["name"],
            "player": e["player"]["name"],
            "x": e["location"][0],
            "y": e["location"][1],
            "shot_type": s["type"]["name"],
            "outcome": s["outcome"]["name"],
            "goal": int(s["outcome"]["name"] == "Goal"),
            "has_frame": e["id"] in frame_ids,
            # new fields
            "body_part": s["body_part"]["name"],
            "technique": s["technique"]["name"],
            "play_pattern": e["play_pattern"]["name"],
            "under_pressure": int(e.get("under_pressure", False)),
            "first_time": int(s.get("first_time", False)),
        })

shots_df = pd.DataFrame(rows)

Path("../data/processed").mkdir(parents=True, exist_ok=True)
shots_df.to_csv("../data/processed/shots_raw.csv", index=False)   

model = shots_df[
    shots_df["has_frame"]
    & (shots_df["period"] != 5)
    & (shots_df["shot_type"] != "Penalty")
]
print(len(shots_df), "->", len(model), "shots | goal rate:", round(model["goal"].mean(), 3))

10831 -> 10233 shots | goal rate: 0.102


In [9]:
import numpy as np

# --- geometry features ---
GOAL_X, GOAL_Y = 120, 40
POST_LO, POST_HI = 36, 44            # goal is 8 yards wide

def goal_angle(x, y):
    # angle between the lines from the shot to each post (law of cosines)
    a = np.hypot(GOAL_X - x, POST_LO - y)
    b = np.hypot(GOAL_X - x, POST_HI - y)
    cos = (a**2 + b**2 - (POST_HI - POST_LO) ** 2) / (2 * a * b)
    return np.arccos(np.clip(cos, -1, 1))

model = model.reset_index(drop=True).copy()
model["distance"] = np.hypot(GOAL_X - model["x"], GOAL_Y - model["y"])
model["angle"] = goal_angle(model["x"], model["y"])
model["y_offset"] = (model["y"] - GOAL_Y).abs()
print(model[["distance", "angle", "y_offset"]].describe().round(2))

       distance     angle  y_offset
count  10233.00  10233.00  10233.00
mean      18.10      0.46      7.70
std        8.16      0.28      5.43
min        0.76      0.00      0.00
25%       11.67      0.28      3.20
50%       17.49      0.37      6.90
75%       23.86      0.56     11.40
max       92.80      2.79     39.60


In [10]:
# --- categorical features ---
def collapse(series, min_count=100):
    # merge rare categories into "other"
    counts = series.value_counts()
    keep = counts[counts >= min_count].index
    return series.where(series.isin(keep), "other")

cats = ["body_part", "technique", "play_pattern"]
for c in cats:
    print(model[c].value_counts(), "\n")      # look before collapsing
    model[c] = collapse(model[c])

onehot = pd.get_dummies(model[cats], dtype=int)

keep_cols = ["shot_id", "match_id", "team", "player", "period", "minute",
             "x", "y", "distance", "angle", "y_offset", "under_pressure", "first_time"]
tabular = pd.concat([model[keep_cols], onehot, model[["goal"]]], axis=1)
tabular.insert(0, "row_idx", range(len(tabular)))   # tensors must use this same row order

tabular.to_csv("../data/processed/tabular.csv", index=False)
print(tabular.shape)

body_part
Right Foot    5158
Left Foot     3270
Head          1751
Other           54
Name: count, dtype: int64 

technique
Normal           7975
Half Volley      1421
Volley            626
Lob                78
Backheel           58
Overhead Kick      40
Diving Header      35
Name: count, dtype: int64 

play_pattern
Regular Play      3527
From Throw In     1862
From Corner       1789
From Free Kick    1788
From Counter       477
From Goal Kick     450
From Keeper        182
From Kick Off      119
Other               39
Name: count, dtype: int64 

(10233, 32)


## match-grouped split
- no match should appear in more than one split

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

tabular = pd.read_csv("../data/processed/tabular.csv")

def assign_split(df, seed):
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    split = pd.Series("train", index=df.index)
    for fold, (_, idx) in enumerate(sgkf.split(df, df["goal"], groups=df["match_id"])):
        if fold == 0:
            split.iloc[idx] = "test"
        elif fold == 1:
            split.iloc[idx] = "val"
    return split

tabular["split"] = assign_split(tabular, seed=42)


# checks
print(tabular.groupby("split").agg(
    matches=("match_id", "nunique"), shots=("goal", "size"),
    goals=("goal", "sum"), goal_rate=("goal", "mean")).round(3))
assert tabular.groupby("match_id")["split"].nunique().max() == 1   # no match in two splits

# save: the split table alone, and tabular with the new column
tabular.groupby("match_id")["split"].first().reset_index() \
       .to_csv("../data/processed/splits.csv", index=False)
tabular.to_csv("../data/processed/tabular.csv", index=False)

       matches  shots  goals  goal_rate
split                                  
test        61   1480    151      0.102
train      296   7290    742      0.102
val         60   1463    146      0.100


## Spatial tensors (CNN)

In [12]:
from matplotlib.path import Path as MplPath

CELL = 2                                   # yards per grid cell
W, H = 120 // CELL, 80 // CELL             # 60 columns (x), 40 rows (y)
CHANNELS = ["attackers", "defenders", "goalkeeper", "shooter", "visible_area", "dist_to_goal"]

# cell centres: used for the visible-area mask and the distance channel
xs = (np.arange(W) + 0.5) * CELL
ys = (np.arange(H) + 0.5) * CELL
XX, YY = np.meshgrid(xs, ys)               # both shape (H, W)
CENTRES = np.c_[XX.ravel(), YY.ravel()]
DIST = np.hypot(120 - XX, 40 - YY) / 120   # static: distance of each cell to goal centre

def cell(loc):
    col = min(max(int(loc[0] // CELL), 0), W - 1)   # clip so players on the line stay in the grid
    row = min(max(int(loc[1] // CELL), 0), H - 1)
    return row, col

def frame_to_tensor(frame):
    t = np.zeros((len(CHANNELS), H, W), dtype=np.float32)
    for p in frame["freeze_frame"]:
        r, c = cell(p["location"])
        if p["actor"]:
            t[3, r, c] += 1                  # the shooter
        elif p["teammate"]:
            t[0, r, c] += 1                  # attackers
        elif p["keeper"]:
            t[2, r, c] += 1                  # opponent goalkeeper
        else:
            t[1, r, c] += 1                  # opponent outfield player
    poly = np.array(frame["visible_area"]).reshape(-1, 2)
    t[4] = MplPath(poly).contains_points(CENTRES).reshape(H, W)
    t[5] = DIST
    return t

In [13]:
tabular = pd.read_csv("../data/processed/tabular.csv")
X = np.zeros((len(tabular), len(CHANNELS), H, W), dtype=np.float16)

for match_id, group in tabular.groupby("match_id"):
    frames = {f["event_uuid"]: f for f in load(DATA / "three-sixty" / f"{match_id}.json")}
    for r in group.itertuples():
        X[r.row_idx] = frame_to_tensor(frames[r.shot_id])

np.savez_compressed("../data/processed/spatial.npz", X=X, row_idx=tabular["row_idx"].values)

Xf = X.astype(np.float32)
print(X.shape)
print("one shooter per tensor:", (Xf[:, 3].sum((1, 2)) == 1).mean())
print("mean attackers / defenders:", Xf[:, 0].sum((1, 2)).mean().round(1), Xf[:, 1].sum((1, 2)).mean().round(1))
print("goalkeeper missing:", (Xf[:, 2].sum((1, 2)) == 0).mean().round(3))

(10233, 6, 40, 60)
one shooter per tensor: 1.0
mean attackers / defenders: 5.0 8.0
goalkeeper missing: 0.062


## text pairs

In [14]:
import random

# --- Phase 3 change: every StatsBomb outcome is mapped explicitly ---
# (before: only Goal/Saved/Blocked were mapped, so "Saved Off Target" and "Saved to Post"
#  fell into "missed the target" -> wrong relevance labels)
OUTCOME = {"Goal": "ended in a goal",
           "Saved": "was saved", "Saved Off Target": "was saved", "Saved to Post": "was saved",
           "Blocked": "was blocked",
           "Off T": "missed the target", "Wayward": "missed the target", "Post": "missed the target"}
# readable outcome words for the DOCUMENT text (before: raw name lowercased -> "off t")
OUTCOME_TEXT = {"Goal": "goal", "Saved": "saved", "Saved Off Target": "saved, off target",
                "Saved to Post": "saved onto the post", "Blocked": "blocked",
                "Off T": "off target", "Wayward": "wayward", "Post": "hit the post"}

def describe_event(e):
    # one short phrase per build-up event, or None to skip it
    who, t = e["player"]["name"], e["type"]["name"]
    if t == "Pass" and "outcome" not in e["pass"]:          # completed passes only
        rec = e["pass"].get("recipient", {}).get("name")
        return f"{who} passes" + (f" to {rec}" if rec else "")
    if t == "Carry":
        (x0, y0), (x1, y1) = e["location"], e["carry"]["end_location"]
        return f"{who} carries the ball" if np.hypot(x1 - x0, y1 - y0) >= 5 else None
    if t == "Dribble" and e["dribble"]["outcome"]["name"] == "Complete":
        return f"{who} dribbles past an opponent"
    if t == "Ball Recovery":
        return f"{who} wins the ball back"
    return None

def shot_attrs(e):
    s, (x, y) = e["shot"], e["location"]
    return {
        "team": e["team"]["name"],
        "body": s["body_part"]["name"].lower(),
        "side": "central" if abs(y - 40) <= 9 else ("left" if y < 40 else "right"),
        "area": "inside the box" if (x >= 102 and 18 <= y <= 62) else "outside the box",
        "outcome": OUTCOME[s["outcome"]["name"]],      # KeyError on an unseen name is deliberate
        "yards": round(float(np.hypot(120 - x, 40 - y))),
    }

def describe_shot(e, buildup, a):
    steps = [d for d in map(describe_event, buildup) if d]
    text = f"{a['team']}, minute {e['minute']}. "
    if steps:
        text += "Build-up: " + "; then ".join(steps) + ". "
    text += (f"{e['player']['name']} shoots with the {a['body']} from {a['yards']} yards "
             f"({a['area']}, {a['side']}). Outcome: {OUTCOME_TEXT[e['shot']['outcome']['name']]}.")
    return text

In [15]:
def make_queries(shot_id, a, n=3):
    rng = random.Random(shot_id)          # same shot -> same queries on every run
    parts = {
        "team": f"by {a['team']}",
        "body": f"with the {a['body']}",
        "side": "from a central position" if a["side"] == "central" else f"from the {a['side']} side",
        "area": f"from {a['area']}",
        "outcome": f"that {a['outcome']}",
    }
    out = []
    for _ in range(n):
        keys = rng.sample(list(parts), k=rng.choice([2, 3]))   # ask about only some attributes
        out.append({"query": "Find a shot " + " ".join(parts[k] for k in keys),
                    "attrs": {k: a[k] for k in keys}})
    return out

In [16]:
import json
meta = tabular.set_index("shot_id")[["row_idx", "split"]]
pairs = []
corpus = []      # Phase 3: one row per shot with its FULL attributes (the retrieval ground truth)

for match_id, group in tabular.groupby("match_id"):
    events = load(DATA / "events" / f"{match_id}.json")
    by_possession = {}
    for e in events:
        by_possession.setdefault(e["possession"], []).append(e)
    wanted = set(group["shot_id"])
    for e in events:
        if e["id"] not in wanted:
            continue
        before = [b for b in by_possession[e["possession"]]
                  if b["index"] < e["index"] and b["team"]["name"] == e["team"]["name"] and "player" in b]
        a = shot_attrs(e)
        doc = describe_shot(e, before[-5:], a)          # last 5 events before the shot
        corpus.append({"shot_id": e["id"], "row_idx": int(meta.loc[e["id"], "row_idx"]),
                       "split": meta.loc[e["id"], "split"], "document": doc, "doc_attrs": a})
        for i, q in enumerate(make_queries(e["id"], a)):
            pairs.append({"pair_id": f"{e['id']}-{i}", "shot_id": e["id"],
                          "row_idx": int(meta.loc[e["id"], "row_idx"]),
                          "split": meta.loc[e["id"], "split"],
                          "query": q["query"], "attrs": q["attrs"], "document": doc})

with open("../data/processed/text_pairs.jsonl", "w", encoding="utf-8") as f:   # utf-8: accents in names
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

with open("../data/processed/text_corpus.jsonl", "w", encoding="utf-8") as f:
    for c in corpus:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(len(corpus), "corpus docs |", len(pairs), "pairs |", pd.Series([p["split"] for p in pairs]).value_counts().to_dict())
for p in random.Random(1).sample(pairs, 2):
    print("\nQ:", p["query"], "\nD:", p["document"])

10233 corpus docs | 30699 pairs | {'train': 21870, 'test': 4440, 'val': 4389}

Q: Find a shot from the left side by Denmark 
D: Denmark, minute 92. Build-up: Pierre-Emile Højbjerg wins the ball back; then Pierre-Emile Højbjerg dribbles past an opponent; then Pierre-Emile Højbjerg carries the ball. Pierre-Emile Højbjerg shoots with the right foot from 23 yards (inside the box, left). Outcome: blocked.

Q: Find a shot that missed the target from a central position 
D: Spain Women's, minute 29. Build-up: Olga  Carmona García carries the ball; then Olga  Carmona García passes to Jennifer Hermoso Fuentes; then Jennifer Hermoso Fuentes passes to Aitana Bonmati Conca. Aitana Bonmati Conca shoots with the left foot from 17 yards (inside the box, central). Outcome: off target.


## Verify phase 1 data

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

tabular = pd.read_csv("../data/processed/tabular.csv")
X = np.load("../data/processed/spatial.npz")["X"]

# 1. tensors and tabular rows line up (the shooter's cell must be where the shot was taken)
shooter_cell = X[:, 3].reshape(len(X), -1).argmax(1)
expected = (tabular["y"] // 2).clip(0, 39).astype(int) * 60 + (tabular["x"] // 2).clip(0, 59).astype(int)
print("rows:", len(tabular), X.shape[0], "| shooter cell matches shot location:", (shooter_cell == expected).mean().round(4))

# 2. no match in more than one split
assert tabular.groupby("match_id")["split"].nunique().max() == 1

# 3. features contain nothing from the outcome, and no missing values
non_features = ["row_idx", "shot_id", "match_id", "team", "player", "period", "minute", "split", "goal"]
feature_cols = [c for c in tabular.columns if c not in non_features]
print(feature_cols)
assert not tabular[feature_cols].isna().any().any()

# 4. text pairs point to the right rows
pairs = [json.loads(l) for l in open("../data/processed/text_pairs.jsonl", encoding="utf-8")]
assert len(pairs) == 3 * len(tabular)
assert all(tabular.loc[p["row_idx"], "shot_id"] == p["shot_id"] for p in pairs)

# 5. a plain logistic regression should be clearly better than chance
tr, te = tabular["split"] == "train", tabular["split"] == "test"
sc = StandardScaler().fit(tabular.loc[tr, feature_cols])
lr = LogisticRegression(max_iter=2000).fit(sc.transform(tabular.loc[tr, feature_cols]), tabular.loc[tr, "goal"])
p_te = lr.predict_proba(sc.transform(tabular.loc[te, feature_cols]))[:, 1]
print("baseline test AUC:", round(roc_auc_score(tabular.loc[te, "goal"], p_te), 3))

# 6. (Phase 3) corpus: one doc per shot, and every query's attrs are a subset of its doc's full attrs
corpus = [json.loads(l) for l in open("../data/processed/text_corpus.jsonl", encoding="utf-8")]
da = {c["shot_id"]: c["doc_attrs"] for c in corpus}
assert len(corpus) == len(tabular) == len(da) == 10233
assert all(da[p["shot_id"]][k] == v for p in pairs for k, v in p["attrs"].items())
print("corpus ok:", len(corpus), "docs")


rows: 10233 10233 | shooter cell matches shot location: 0.9992
['x', 'y', 'distance', 'angle', 'y_offset', 'under_pressure', 'first_time', 'body_part_Head', 'body_part_Left Foot', 'body_part_Right Foot', 'body_part_other', 'technique_Half Volley', 'technique_Normal', 'technique_Volley', 'technique_other', 'play_pattern_From Corner', 'play_pattern_From Counter', 'play_pattern_From Free Kick', 'play_pattern_From Goal Kick', 'play_pattern_From Keeper', 'play_pattern_From Kick Off', 'play_pattern_From Throw In', 'play_pattern_Regular Play', 'play_pattern_other']
baseline test AUC: 0.78
corpus ok: 10233 docs


## Phase 3: retrieval ground truth
A document is **relevant** to a query if it satisfies ALL the attributes in the query's `attrs`. This cell counts how many relevant docs each query has in its own split's corpus (the pool that will be searched at evaluation time).

In [18]:
import collections
corpus = [json.loads(l) for l in open("../data/processed/text_corpus.jsonl", encoding="utf-8")]
pairs  = [json.loads(l) for l in open("../data/processed/text_pairs.jsonl", encoding="utf-8")]

by_split = collections.defaultdict(list)
for c in corpus:
    by_split[c["split"]].append(c["doc_attrs"])

def n_relevant(attrs, docs):
    return sum(all(d[k] == v for k, v in attrs.items()) for d in docs)

for split in ["val", "test"]:
    docs = by_split[split]
    counts = pd.Series([n_relevant(p["attrs"], docs) for p in pairs if p["split"] == split])
    print(split, "| corpus size:", len(docs), "| queries:", len(counts))
    print(counts.describe(percentiles=[.1, .5, .9]).round(1).to_string(), "\n")
    print("queries with >= 10 relevant docs:", (counts >= 10).mean().round(3), "\n")

val | corpus size: 1463 | queries: 4389
count    4389.0
mean      103.2
std       130.2
min         1.0
10%         3.0
50%        46.0
90%       276.0
max       586.0 

queries with >= 10 relevant docs: 0.698 

test | corpus size: 1480 | queries: 4440
count    4440.0
mean      107.0
std       141.3
min         1.0
10%         3.0
50%        42.0
90%       287.0
max       636.0 

queries with >= 10 relevant docs: 0.696 

